# 8. 텍스트 수집, 전처리와 토큰화

| 순서 | 내용 |
|------|------|
| 1 | 텍스트마이닝 개요 |
| 2 | 텍스트 데이터 수집: 웹 크롤링 |
| 3 | 데이터 정제 |
| 4 | 정규표현식 |
| 5 | 토큰화와 형태소 분석 |
| 6 | 리뷰 키워드 확인 |


# 텍스트마이닝 개요

## 1. 텍스트마이닝이란?

### 1.1 텍스트마이닝의 정의
- 비정형 텍스트에서 유용한 정보, 패턴, 주제, 감정, 관계를 추출하는 데이터 분석 방법입니다.
- 자연어 처리(NLP)는 컴퓨터가 언어를 다루게 하는 기술 영역이고, 텍스트마이닝은 그 기술을 활용해 비즈니스·사회·연구 문제를 분석하는 응용 흐름에 가깝습니다.
- 실무에서는 `수집 -> 정제 -> 토큰화 -> 벡터화/임베딩 -> 분석/모델링 -> 해석` 순서로 진행되는 경우가 많습니다.

### 1.2 일반적인 데이터 분석과의 차이
| 구분 | 정형 데이터 분석 | 텍스트마이닝 |
|------|------------------|--------------|
| 원본 형태 | 숫자, 범주, 날짜처럼 열 구조가 명확함 | 문장, 댓글, 기사, 문서처럼 자유롭게 작성됨 |
| 주요 전처리 | 결측치, 이상치, 스케일 조정, 인코딩 | 불필요한 문자 제거, 표기 통일, 토큰화, 형태소 분석 |
| 특징 생성 | 기존 열을 변환하거나 조합 | 단어, n-gram, 키워드, 문서 벡터, 임베딩 생성 |
| 해석 관점 | 변수와 목표값의 관계 | 단어·문맥·주제·감정·의견의 흐름 |
| 어려운 점 | 데이터 품질, 변수 선택, 모델 평가 | 중의성, 신조어, 오타, 문맥, 도메인 용어 |

### 1.3 주요 활용 사례
- **리뷰 분석**: 상품·서비스 리뷰에서 만족 요인, 불만, 개선 요구, 감성 점수를 찾습니다.
- **뉴스 분석**: 기사 제목과 본문을 모아 이슈 흐름, 언론사별 관점, 주요 인물·기업·사건을 추적합니다.
- **여론 분석**: 댓글, 게시글, 설문 응답에서 찬반 의견, 관심 주제, 정책 반응을 파악합니다.
- **키워드 분석**: 빈도, TF-IDF, 동시출현, 워드클라우드로 문서 집합의 핵심 단어를 요약합니다.
- **챗봇/RAG**: 문서를 잘게 나누고 임베딩으로 검색한 뒤, 관련 근거를 언어모델에 전달해 답변을 생성합니다.

### 1.4 기본 처리 흐름
텍스트 분석은 보통 `수집 -> 정제 -> 토큰화 -> 수치화 -> 분석` 순서로 진행합니다. 이 노트북에서는 앞의 세 단계인 수집, 정제, 토큰화를 다룹니다.


## 2. 텍스트 데이터 다루기

### 2.1 텍스트 데이터 수집: 웹 크롤링

텍스트마이닝은 분석할 텍스트를 확보하는 일에서 시작합니다. 직접 설문을 만들 수도 있지만, 뉴스·리뷰·게시글처럼 이미 웹에 공개된 텍스트를 수집해서 분석하는 경우도 많습니다.

이번 실습에서는 **Playwright**로 브라우저를 직접 열고, 화면 안의 요소를 선택해 텍스트를 수집합니다. Playwright를 사용하면 실제 브라우저 동작, 기다림, iframe 선택자를 함께 다룰 수 있습니다.

데이터 분석 흐름에서 크롤링은 다음 단계의 가장 앞에 있습니다.

`수집 -> 정제 -> 토큰화 -> 벡터화 -> 모델링`

<img src="image/web_crawling_background.svg" width="780">


#### 웹에서 텍스트를 가져올 때 보는 기본 요소

웹 크롤링을 처음 할 때는 모든 웹 기술을 다 이해할 필요가 없습니다.  
이번 실습에서는 **주소(URL)를 열고, 화면에 있는 표나 목록에서 필요한 글자를 가져오는 것**에 집중합니다.

| 요소 | 역할 | 크롤링에서 확인할 점 |
|------|------|----------------------|
| 브라우저 | 웹 페이지를 열어 화면에 보여줌 | 개발자도구로 제목, 날짜, 표 위치를 확인함 |
| URL | 웹 페이지의 주소 | 종목 코드, 검색어, 페이지 번호처럼 바뀌는 값을 찾음 |
| HTTP/HTTPS | 브라우저와 서버가 데이터를 주고받는 방식 | 페이지가 정상으로 열리는지 확인함 |
| 웹 서버 | 요청받은 페이지나 데이터를 보내줌 | 같은 주소를 코드에서도 열 수 있음 |
| HTML | 페이지의 내용과 구조 | 제목, 표, 링크, 날짜가 들어 있는 태그를 찾음 |
| CSS | 화면 스타일을 지정하는 정보 | `class` 이름이 선택자 힌트가 되기도 함 |
| JavaScript | 화면을 나중에 채우거나 버튼 동작을 처리함 | 데이터가 늦게 보이면 기다림이 필요할 수 있음 |

예를 들어 아래 주소에서는 `code=005930`이 종목 코드, `page=1`이 페이지 번호입니다.

```text
https://finance.naver.com/item/sise_day.naver?code=005930&page=1
```

텍스트 수집 실습에서는 이런 값을 바꿔가며 여러 페이지의 뉴스 제목, 언론사, 날짜를 모읍니다.


#### HTML 태그 읽기

HTML은 웹 페이지의 뼈대입니다. 태그가 중첩되면서 문서 구조를 만들고, 각 태그 안에 텍스트나 링크가 들어갑니다.

```html
<tr class="price-row">
  <td class="date">2026.05.17</td>
  <td class="close">70,000</td>
  <td class="volume">12,345,678</td>
</tr>
```

위 HTML에서 `tr`은 표의 한 행, `td`는 표의 한 칸입니다.  
`class="date"`처럼 태그에 붙은 정보는 **속성(attribute)** 입니다.

크롤링은 결국 화면 안에서 **반복되는 구조**를 찾는 일입니다.  
뉴스 목록은 여러 행이 반복되고, 각 행 안에 제목, 언론사, 날짜, 링크가 들어 있습니다.

Playwright에서는 이런 반복 요소를 `locator()`로 찾습니다.

```python
rows = page.locator("tr.price-row")
first_row = rows.nth(0)
date = await first_row.locator("td.date").inner_text()
```

처음에는 선택자를 완벽히 외우기보다, 개발자도구에서 반복되는 행과 그 안의 제목 태그를 찾는 연습에 집중하면 됩니다.


#### 개발자도구로 수집 위치 찾기

크롤링 코드를 바로 작성하기보다 브라우저 개발자도구로 먼저 확인하면 시행착오가 줄어듭니다. Chrome 기준으로는 페이지에서 마우스 오른쪽 클릭 후 **검사**를 누르면 됩니다.

이번 실습에서는 주로 **Elements 탭**만 확인합니다.

- 화면에 보이는 글자가 HTML의 어느 태그에 들어 있는지 확인합니다.
- 제목, 날짜, 표의 행처럼 반복되는 단위를 찾습니다.
- 태그 이름, `class`, `id`를 보고 선택자 힌트를 얻습니다.
- 목록에서 한 건을 나타내는 반복 단위와 그 안의 제목·날짜·링크 위치를 확인합니다.

개발자도구에서 CSS selector를 복사할 수도 있지만, 너무 긴 selector는 페이지가 조금만 바뀌어도 깨질 수 있습니다.  
`body > div > table > tbody > tr:nth-child(3)`처럼 위치에 의존하는 선택자보다 `table.type2 tr`, `td.title a.tit`처럼 의미 있는 태그와 클래스 조합을 쓰는 편이 안정적입니다.


#### 데이터 수집 원리

크롤링은 사람이 브라우저로 하는 일을 코드로 반복하는 과정입니다.

<img src="image/web_crawling_flow.svg" width="760">

기본 순서는 다음과 같습니다.

1. 수집할 항목을 정합니다. 예: 뉴스 제목, 언론사, 날짜, 링크
2. 브라우저 개발자도구나 Playwright로 반복되는 HTML 구조를 확인합니다.
3. `page.goto()`로 수집할 페이지를 엽니다.
4. `locator()`로 뉴스 행과 제목·언론사·날짜 위치를 찾습니다.
5. `count()`, `nth()`, `inner_text()`, `get_attribute()`로 값을 하나씩 꺼냅니다.
6. 페이지 번호, 검색어, 종목 코드 같은 값을 바꿔가며 반복 수집합니다.
7. 누락값, 중복, 날짜 범위를 확인한 뒤 데이터프레임이나 CSV로 저장합니다.

이번 실습에서는 최신 웹 기술을 깊게 다루기보다, **화면에 보이는 표와 목록에서 텍스트를 가져오는 감각**을 익히는 데 집중합니다.

| 도구 | 적합한 상황 | 장점 | 주의할 점 |
|------|-------------|------|-----------|
| `Playwright` | 브라우저로 페이지를 열고 표나 목록을 확인해야 할 때 | 실제 브라우저처럼 실행되고 기다림 처리가 편함 | 최초 브라우저 설치가 필요함 |

크롤링 코드로 정리할 때는 다음 요소를 확인합니다.

- `page.goto(...)`: 수집할 페이지로 이동
- `locator(...)`: 반복되는 항목과 필요한 하위 요소 선택
- `inner_text()`: 화면에 보이는 텍스트 가져오기
- `get_attribute("href")`: 링크 주소 가져오기
- `wait_for()`: 데이터가 나타날 때까지 기다리기

크롤링할 때는 사이트의 `robots.txt`와 이용약관을 확인하고, 너무 빠르게 반복 요청하지 않아야 합니다. 개인정보, 저작권, 유료 콘텐츠도 함부로 수집하거나 재배포하면 안 됩니다.


#### 실습: 네이버 금융 종목 뉴스 수집

이번 실습은 함수로 감싸지 않고, 셀을 하나씩 실행하면서 확인합니다.  
각 단계에서 변수 내용을 직접 확인하면 어디에서 값이 비었는지, 선택자가 맞는지 디버깅하기 쉽습니다.

수집 대상 URL은 종목 코드와 페이지 번호만 바꾸면 됩니다.

```text
https://finance.naver.com/item/news.naver?code=005930&page=1
```

네이버 금융의 종목 뉴스 표는 페이지 안의 `news` iframe에 들어 있습니다. Playwright에서는 `frame_locator()`로 iframe 안의 표를 선택합니다.

| 항목 | 선택자 |
|------|--------|
| 뉴스 iframe | `iframe[name='news']` |
| 뉴스 표 | `table.type5` |
| 뉴스 행 | `tr` |
| 제목 | `td.title a.tit` |
| 언론사 | `td.info` |
| 날짜 | `td.date` |


In [ ]:
import asyncio
import sys
import threading
from urllib.parse import urljoin

import pandas as pd
from playwright.async_api import async_playwright


class PlaywrightNotebookRunner:
    def __init__(self):
        if sys.platform.startswith("win") and hasattr(asyncio, "ProactorEventLoop"):
            self.loop = asyncio.ProactorEventLoop()
        else:
            self.loop = asyncio.new_event_loop()

        self.thread = threading.Thread(target=self._run_loop, daemon=True)
        self.thread.start()

    def _run_loop(self):
        asyncio.set_event_loop(self.loop)
        self.loop.run_forever()

    def run(self, awaitable):
        future = asyncio.run_coroutine_threadsafe(awaitable, self.loop)
        return future.result()

    def stop(self):
        self.loop.call_soon_threadsafe(self.loop.stop)


runner = PlaywrightNotebookRunner()
run = runner.run

code = "005930"  # 삼성전자
page_no = 1

url = f"https://finance.naver.com/item/news.naver?code={code}&page={page_no}"
url


##### 1단계. 브라우저를 열고 페이지 이동하기

노트북에서는 Playwright 작업을 `run(...)`으로 실행합니다.  
처음에는 브라우저 창을 실제로 띄워서 어떤 페이지가 열리는지 확인합니다. 화면을 띄우지 않고 실행하려면 `headless=False`를 `True`로 바꾸면 됩니다.


In [ ]:
playwright = run(async_playwright().start())
browser = run(playwright.chromium.launch(headless=False))
page = run(browser.new_page(locale="ko-KR"))

run(page.goto(url, wait_until="load"))

news_frame = page.frame_locator("iframe[name='news']")
news_table = news_frame.locator("table.type5").first
run(news_table.wait_for(timeout=10_000))


##### 2단계. 뉴스 행 개수 확인하기

표 전체를 한 번에 데이터프레임으로 만들기 전에, 먼저 반복되는 행이 몇 개인지 확인합니다.


In [ ]:
news_rows = news_table.locator("tr")
row_count = run(news_rows.count())
row_count


##### 3단계. 실제 뉴스가 들어 있는 행 하나 확인하기

표에는 구분선이나 빈 행도 섞여 있을 수 있습니다.  
제목 링크가 들어 있는 첫 번째 행을 찾아서 텍스트를 직접 확인합니다.


In [ ]:
sample_row = None
sample_index = None

for i in range(row_count):
    row = news_rows.nth(i)
    title_links = row.locator("td.title a.tit")

    if run(title_links.count()) > 0:
        sample_row = row
        sample_index = i
        break

assert sample_row is not None, "뉴스 행을 찾지 못했습니다. 선택자를 다시 확인하세요."
sample_index, run(sample_row.inner_text())


##### 4단계. 한 행에서 제목, 언론사, 날짜, 링크 꺼내기

한 건을 제대로 꺼낼 수 있으면, 같은 방법을 반복해서 여러 건을 모을 수 있습니다.


In [ ]:
title_link = sample_row.locator("td.title a.tit").first
source_cell = sample_row.locator("td.info").first
date_cell = sample_row.locator("td.date").first

href = run(title_link.get_attribute("href"))

sample_news = {
    "title": run(title_link.inner_text()).strip(),
    "source": run(source_cell.inner_text()).strip() if run(source_cell.count()) else None,
    "date": run(date_cell.inner_text()).strip() if run(date_cell.count()) else None,
    "link": urljoin("https://finance.naver.com", href or ""),
    "page": page_no,
}

sample_news


##### 5단계. 현재 페이지의 뉴스 모으기

이제 한 행에서 값을 꺼내던 코드를 반복문으로 감싸 현재 페이지의 뉴스 목록을 만듭니다.


In [ ]:
rows = []

for i in range(row_count):
    row = news_rows.nth(i)
    title_links = row.locator("td.title a.tit")

    if run(title_links.count()) == 0:
        continue

    title_link = title_links.first
    source_cell = row.locator("td.info").first
    date_cell = row.locator("td.date").first
    href = run(title_link.get_attribute("href"))

    rows.append({
        "title": run(title_link.inner_text()).strip(),
        "source": run(source_cell.inner_text()).strip() if run(source_cell.count()) else None,
        "date": run(date_cell.inner_text()).strip() if run(date_cell.count()) else None,
        "link": urljoin("https://finance.naver.com", href or ""),
        "page": page_no,
    })

page_df = pd.DataFrame(rows).drop_duplicates(subset=["title", "date", "link"]).reset_index(drop=True)
page_df.head()


##### 6단계. 페이지 번호를 바꿔 여러 페이지 수집하기

현재 페이지 수집이 잘 되면, 페이지 번호만 바꿔가며 같은 과정을 반복합니다.


In [ ]:
code = "005930"
start_page = 1
end_page = 2

rows = []

for page_no in range(start_page, end_page + 1):
    url = f"https://finance.naver.com/item/news.naver?code={code}&page={page_no}"
    run(page.goto(url, wait_until="load"))

    news_frame = page.frame_locator("iframe[name='news']")
    news_table = news_frame.locator("table.type5").first
    run(news_table.wait_for(timeout=10_000))

    news_rows = news_table.locator("tr")
    row_count = run(news_rows.count())

    for i in range(row_count):
        row = news_rows.nth(i)
        title_links = row.locator("td.title a.tit")

        if run(title_links.count()) == 0:
            continue

        title_link = title_links.first
        source_cell = row.locator("td.info").first
        date_cell = row.locator("td.date").first
        href = run(title_link.get_attribute("href"))

        rows.append({
            "title": run(title_link.inner_text()).strip(),
            "source": run(source_cell.inner_text()).strip() if run(source_cell.count()) else None,
            "date": run(date_cell.inner_text()).strip() if run(date_cell.count()) else None,
            "link": urljoin("https://finance.naver.com", href or ""),
            "page": page_no,
        })

    run(page.wait_for_timeout(500))

news_df = pd.DataFrame(rows).drop_duplicates(subset=["title", "date", "link"]).reset_index(drop=True)
news_df.head()


##### 7단계. 브라우저 닫기

수집이 끝나면 브라우저와 Playwright를 종료합니다.


In [ ]:
run(browser.close())
run(playwright.stop())
runner.stop()


### 문제. 수집 대상 바꾸기

위의 여러 페이지 수집 셀에서 아래 조건으로 값을 바꿔 `news_df`를 다시 만드세요.

조건:
- 종목 코드: `035720` (카카오)
- 수집 범위: 1~3페이지
- 결과 열: `title`, `source`, `date`, `link`, `page`

<details>
<summary>정답 보기</summary>

```python
code = "035720"
start_page = 1
end_page = 3
```

위 세 값을 바꾼 뒤, 여러 페이지 수집 셀을 다시 실행하면 됩니다.

</details>


### 2.2 데이터 정제 (Cleaning)
텍스트 데이터에는 사람이 쓰면서 생긴 **불필요한 잡음(noise)** 이 많이 섞여 있습니다.  
이런 잡음을 제거하지 않으면 모델이 쓸데없는 패턴까지 학습해서 성능이 떨어질 수 있습니다.  

#### 정제 대상
- 불필요한 특수문자, 구두점  
- HTML 태그  
- 중복된 공백, 줄바꿈  
- 대소문자 혼재  
- 불용어(stopwords)


####  다양한 예시

1) **특수문자 제거**
```
원문: "안녕??? 오늘 날씨 진짜 좋다~~~^^"
정제: "안녕 오늘 날씨 진짜 좋다"
```

2) **HTML 태그 제거**
```
원문: "<div>이 영화 <b>정말</b> 최고!!!</div>"
정제: "이 영화 정말 최고"
```

3) **중복 공백/개행 제거**
```
원문: "오늘은   점심에    김밥을   먹었다. \n\n 내일도 김밥?"
정제: "오늘은 점심에 김밥을 먹었다. 내일도 김밥?"
```

4) **대소문자 통일**
```
원문: "Apple is Better than apple."
정제(소문자화): "apple is better than apple."
```

5) **불용어 제거** *(전통 ML·IR에서 선택적 / **LLM 파이프라인에선 보통 사용하지 않음**)*  
- **불용어(Stopwords)**: 문장에서 자주 등장하지만 분류·검색 성능에 **한정적으로만** 기여하는 단어 집합.  
  예: 국문 — “나는/그리고/하지만/오늘/에서 …”, 영문 — “the/and/of/to …”  
- 전통 BoW/TF-IDF, IR 인덱싱에서 **차원 축소·노이즈 감소** 목적으로 **선택적** 사용.  
- LLM 파이프라인(사전학습/미세조정/추론)에서는 **보존**하는 것이 일반적.

```
원문: "나는 오늘 점심으로 김밥을 먹었다."
정제(불용어 제거 예 — 전통 ML/IR용): "오늘 점심 김밥 먹었다"
```


👉 이렇게 정제를 통해 **텍스트를 더 단순하고 의미 중심적으로 바꿔야** 이후 단계(토큰화, 임베딩 등)가 효과적으로 작동합니다.

#### 목적에 따라 전처리 전략은 달라집니다

전처리가 항상 좋은 것은 아닙니다. **무엇을 위해 사용하는가**에 따라 달라집니다.

| 목적 | 전처리 필요성 | 이유 |
|------|----------------|------|
| **전통 워드클라우드, TF-IDF 분석** | 높음 | 단순 빈도 기반이므로 노이즈 단어가 시각화를 망침 |
| **전통 ML(감성분석, 분류 등)** | 중간 | 불용어 제거·토큰 정규화가 도움되기도 함 |
| **LLM 파이프라인(사전학습·미세조정·RAG·챗봇)** | 낮음 | LLM은 문맥 기반 모델이므로 불용어·기호도 의미 구조 해석에 필요 |
| **언어학적 분석(구문·어휘 다양성)** | 매우 낮음 | 원문 보존이 핵심 |

즉, **좋은 전처리란, 모든 것을 없애는 것이 아니라 목적에 맞게 적절히 다듬는 것**입니다.

#### 실습: 전처리한 텍스트로 워드클라우드 만들기

워드클라우드는 단어 빈도를 글자 크기로 보여주는 시각화입니다.  
조사, 숫자, 특수문자 같은 노이즈를 정리한 뒤에 만들면 핵심 단어가 훨씬 잘 보입니다.

아래는 같은 방식으로 만들 수 있는 워드클라우드 예시입니다.

<img src="image/wordcloud_example.svg" width="760">



In [ ]:
import re  # 정규표현식을 사용하기 위한 모듈입니다.
from collections import Counter  # 단어 빈도를 세는 도구입니다.

import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.
import matplotlib.font_manager as fm  # 시스템 폰트를 찾기 위한 도구입니다.
from wordcloud import WordCloud  # 단어 빈도를 워드클라우드로 시각화합니다.

texts = [
    "배송 지연으로 고객 불만이 증가했습니다. 배송 안내가 더 필요합니다.",
    "환불 요청이 많아 상담 대기 시간이 길어졌습니다.",
    "상품 품질은 좋지만 포장 불량과 배송 지연이 반복됩니다.",
    "빠른 환불 처리와 친절한 상담이 고객 만족을 높였습니다.",
    "배송 상태 알림과 교환 절차 안내를 개선해야 합니다."
]  # 분석할 예시 문장입니다.

stopwords = {"이", "가", "을", "를", "은", "는", "과", "와", "으로", "더"}  # 제외할 단어입니다.

def clean_text(text):
    text = re.sub(r"[^가-힣a-zA-Z\s]", " ", text)  # 한글/영문/공백만 남깁니다.
    return re.sub(r"\s+", " ", text).strip()  # 여러 공백을 하나로 정리합니다.

tokens = []
for text in texts:
    cleaned = clean_text(text)  # 문장별로 노이즈를 제거합니다.
    tokens.extend([word for word in cleaned.split() if word not in stopwords and len(word) > 1])  # 의미 단어만 남깁니다.

freq = Counter(tokens)  # 단어별 등장 횟수를 계산합니다.
freq.most_common(10)  # 가장 자주 나온 단어를 확인합니다.

font_candidates = ["Noto Sans CJK KR", "NanumGothic", "Malgun Gothic", "AppleGothic"]  # 한글 폰트 후보입니다.
font_path = next(
    (font.fname for font in fm.fontManager.ttflist if any(name in font.name for name in font_candidates)),
    None
)  # 사용 가능한 한글 폰트 경로를 찾습니다.

wc = WordCloud(
    font_path=font_path,
    width=900,
    height=450,
    background_color="white",
    colormap="viridis"
).generate_from_frequencies(freq)  # 단어 빈도로 워드클라우드를 만듭니다.

plt.figure(figsize=(10, 5))  # 그래프 크기와 도화지를 설정합니다.
plt.imshow(wc, interpolation="bilinear")  # 워드클라우드 이미지를 표시합니다.
plt.axis("off")  # 축을 숨깁니다.
plt.show()  # 그래프를 화면에 출력합니다.


#### 문제 1. 대소문자 통일
문장 `"Machine Learning is FUN and Useful."`을 모두 소문자로 바꿔보세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

text = "Machine Learning is FUN and Useful."  # 실습할 문자열을 준비합니다.
print(text.lower())  # 결과를 화면에 출력합니다.
# 출력: "machine learning is fun and useful."

```
</details>

In [ ]:
# 여기에 작성하세요
text = "Machine Learning is FUN and Useful."

#### 문제 2. 불용어 제거
문장 `"나는 오늘 아침에 학교에 갔다."`에서 불용어 ["나는", "오늘", "에"]를 제거해보세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

text = "나는 오늘 아침에 학교에 갔다."  # 실습할 문자열을 준비합니다.
stopwords = ["나는", "오늘", "에"]  # 분석에서 제외할 불용어 목록입니다.

for word in stopwords:  # 값을 하나씩 꺼내 반복합니다.
    text = text.replace(word, '')  # 실습할 문자열을 준비합니다.
    
text.strip()  # 양쪽 공백을 제거합니다.
print(text)  # 결과를 화면에 출력합니다.
# 출력: "아침에 학교에 갔다."

```
</details>

In [ ]:
# 여기에 작성하세요
text = "나는 오늘 아침에 학교에 갔다."
stopwords = ["나는", "오늘", "에"]

### 2.3 정규표현식(Regex)
정규표현식(Regular Expression, **regex**)은 **문자열에서 특정 패턴을 찾고/바꾸고/분리**하는 강력한 도구입니다.  
전자영수증에서 숫자만 뽑아내고, 로그에서 IP만 추출하고, 텍스트 노이즈를 빠르게 정리하는 등 **대량의 텍스트 처리 자동화**에 핵심적으로 쓰입니다.


#### 왜 Regex를 쓰나요?
- **일관된 패턴**을 한 번에 처리 (이메일, URL, 날짜, 숫자, 해시태그 등)
- **간결한 코드**로 복잡한 문자열 조작 수행
- 전처리(클리닝) 단계에서 **재사용 가능한 규칙**으로 품질 유지


#### 핵심 문법(요약)
- **문자클래스**: `\d`(숫자), `\D`(숫자 아님), `\w`(단어문자: [A-Za-z0-9_]), `\s`(공백)  
- **반복/수량자**: `*`(0+), `+`(1+), `?`(0 or 1), `{m,n}`(m~n회)  
- **그룹/선택**: `( )`(캡처 그룹), `(?: )`(비캡처), `|`(OR)  
- **앵커**: `^`(문자열/행 시작), `$`(문자열/행 끝), `\b`(단어 경계)  
- **플래그**: `re.I`(대소문자 무시), `re.M`(멀티라인: ^,$를 행 단위로), `re.S`(dot이 개행 포함)  
- **탐욕/게으름**: `.*`(탐욕적), `.*?`(게으름; 가능한 한 짧게)

> **주의**: 파이썬에서는 `r"..."` **원시 문자열**을 사용해 백슬래시 이스케이프를 피하세요.


#### 정규표현식 주요 패턴 표

| 패턴 | 의미 | 예시 | 매칭 결과 |
|------|------|------|-----------|
| `.` | 임의의 한 문자(개행 제외) | `a.c` | `abc`, `axc` |
| `^` | 문자열/행의 시작 | `^Hi` | `"Hi there"` |
| `$` | 문자열/행의 끝 | `end$` | `"the end"` |
| `\d` | 숫자 (0–9) | `\d{3}` | `123`, `007` |
| `\D` | 숫자가 아닌 문자 | `\D+` | `"abc"`, `"--"` |
| `\w` | 단어문자 `[A-Za-z0-9_]` | `\w+` | `"hello"`, `"Python3"` |
| `\W` | 단어문자가 아닌 것 | `\W+` | `"!!"`, `" "` |
| `\s` | 공백 (스페이스, 탭, 개행) | `a\sb` | `"a b"` |
| `\S` | 공백이 아닌 문자 | `\S+` | `"text"`, `"123"` |
| `*` | 0회 이상 반복 | `ab*` | `"a"`, `"ab"`, `"abbb"` |
| `+` | 1회 이상 반복 | `ab+` | `"ab"`, `"abbb"` |
| `?` | 0회 또는 1회 | `ab?` | `"a"`, `"ab"` |
| `{m,n}` | m~n회 반복 | `\d{2,4}` | `99`, `2025` |
| `( )` | 그룹화 / 캡처 | `(ab)+` | `"ab"`, `"abab"` |
| `(?: )` | 비캡처 그룹 | `(?:ab)+` | `"abab"` |
| `|` | OR 선택 | `cat|dog` | `"cat"`, `"dog"` |
| `\b` | 단어 경계 | `\bcat\b` | `"cat"` (단어 단독일 때) |
| `(?i)` | 대소문자 무시 플래그 | `(?i)abc` | `"abc"`, `"ABC"` |

#### 파이썬 정규표현식 함수 요약

| 함수 | 설명 | 예시 코드 | 결과 |
|------|------|-----------|------|
| `re.match(pattern, string)` | 문자열 **처음부터** 패턴 매칭 | `re.match(r"\d+", "123abc")` | `<Match '123'>` |
| `re.search(pattern, string)` | 문자열 전체에서 **처음 매칭되는 패턴** 찾기 | `re.search(r"\d+", "abc123xyz")` | `<Match '123'>` |
| `re.findall(pattern, string)` | **모든 매칭 결과**를 리스트로 반환 | `re.findall(r"\d+", "a12 b34 c56")` | `['12', '34', '56']` |
| `re.finditer(pattern, string)` | 모든 매칭 결과를 **이터레이터(객체)** 로 반환 | `[m.group() for m in re.finditer(r"\d+", "a12 b34")]` | `['12', '34']` |
| `re.sub(pattern, repl, string)` | 패턴을 다른 문자열로 **치환** | `re.sub(r"\d+", "#", "ID123")` | `"ID#"` |
| `re.split(pattern, string)` | 패턴 기준으로 문자열 **분리** | `re.split(r"\s+", "a b   c")` | `['a', 'b', 'c']` |
| `re.fullmatch(pattern, string)` | 문자열 전체가 패턴과 **완전히 일치**할 때 매칭 | `re.fullmatch(r"\d{3}", "123")` | `<Match '123'>` |
| `re.compile(pattern)` | 정규표현식을 객체로 컴파일 (재사용 최적화) | `p = re.compile(r"\d+")`<br>`p.findall("1a2b3")` | `['1','2','3']` |

> ⚠️ `match`는 문자열의 시작 부분만 확인, `search`는 전체 탐색을 수행한다는 점이 중요합니다.

#### 자주 쓰는 패턴 예시

In [ ]:
### 1) 숫자/기호 제거 + 공백 정규화

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "오늘은 2025년 9월 10일!!! 날씨   정말 좋다   ^^"
no_digits = re.sub(r"\d+", "", text)               # 숫자 제거
no_punct  = re.sub(r"[^\w\s가-힣]", " ", no_digits) # 기호 제거(한글/영문/숫자/공백만 남김)
cleaned   = re.sub(r"\s+", " ", no_punct).strip()   # 다중 공백 → 단일 공백
print(cleaned)  # "오늘은 년 월 일 날씨 정말 좋다"

In [ ]:
### 2) 이메일 추출

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "문의: admin@example.com, 혹은 support@my-site.co.kr 로 연락주세요."
emails = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}", text)
print(emails)  # ['admin@example.com', 'support@my-site.co.kr']

In [ ]:
### 3) URL 제거 (http/https)

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "공식 문서: https://docs.python.org 참고, 우리 블로그 http://example.com/blog 도 봐요."
no_url = re.sub(r"https?://\S+", "", text).strip()  # 정규표현식으로 문자열을 치환합니다.
print(no_url)  # "공식 문서:  참고, 우리 블로그  도 봐요."

In [ ]:
### 4) 한국 전화번호 마스킹

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "연락처: 010-1234-5678 / 02-345-6789"
masked = re.sub(r"\b(\d{2,3})-(\d{3,4})-(\d{4})\b", r"\1-****-****", text)  # 정규표현식으로 문자열을 치환합니다.
print(masked)  # "연락처: 010-****-**** / 02-****-****"

In [ ]:
### 5) 멀티라인에서 행 시작/끝 활용 (`re.M`)

import re  # 정규표현식을 사용하기 위한 모듈입니다.
log = "OK: step1\nERROR: step2 failed\nOK: step3"
errors = re.findall(r"^ERROR:.*$", log, flags=re.M)
print(errors)  # ['ERROR: step2 failed']

In [ ]:
### 6) 탐욕 vs 게으름 (HTML 태그 사이 내용 캡처 예시)

import re  # 정규표현식을 사용하기 위한 모듈입니다.
html = "<p>첫째</p><p>둘째</p>"
greedy = re.findall(r"<p>.*</p>", html)      # 탐욕적: 한 방에 다 먹음
lazy   = re.findall(r"<p>.*?</p>", html)     # 게으름: 가능한 짧게 두 개로 나눔
print(greedy)  # ['<p>첫째</p><p>둘째</p>']
print(lazy)    # ['<p>첫째</p>', '<p>둘째</p>']

> **HTML 파싱은 정규표현식만으로 완벽히 처리하기 어렵습니다.**  
> 웹 페이지처럼 태그 구조가 있는 자료는 Playwright `locator()`처럼 구조를 읽는 도구로 다루는 편이 안정적입니다.


#### 문제 3. 숫자와 기호 제거 + 공백 정규화
문자열에서 **숫자/특수기호를 제거**하고, **다중 공백을 하나로** 바꾼 문자열을 출력하세요.  
문자/숫자/공백/한글만 남기도록 하세요.

<details> <summary>정답 보기</summary>

```python 
import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "정가: 19,800원!!! ★★ 대박 할인 30% ★★  (한정 수량)"
tmp = re.sub(r"\d+", "", text)                         # 숫자 제거
tmp = re.sub(r"[^\w\s가-힣]", " ", tmp)                # 기호 제거
result = re.sub(r"\s+", " ", tmp).strip()             # 공백 정규화
print(result)  # "정가 원 대박 할인 한정 수량"
```
</details>



In [ ]:
# 여기에 작성하세요
import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "정가: 19,800원!!! ★★ 대박 할인 30% ★★  (한정 수량)"

#### 문제 4. 이메일만 추출하기
문장에서 모든 이메일을 찾아 **리스트**로 반환하세요.

<details> <summary>정답 보기</summary>

```python 
import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "메일: kim.ai@univ.ac.kr; 홍보: sales-team@example.com; 오류: bug+test@my.io"
emails = re.findall(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}", text)
print(emails)  # 문자열을 정수로 바꿉니다.
# ['kim.ai@univ.ac.kr', 'sales-team@example.com', 'bug+test@my.io']
```
</details>



In [ ]:
# 여기에 작성하세요
text = "메일: kim.ai@univ.ac.kr; 홍보: sales-team@example.com; 오류: bug+test@my.io"

#### 문제 5. URL 제거하기
문장에서 **http/https URL을 모두 제거**하세요. (도메인 뒤 공백 정리 포함)

<details> <summary>정답 보기</summary>

```python 
import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "문서: https://a.b/c?x=1  블로그: http://blog.com/post  끝."
no_url = re.sub(r"https?://\S+", "", text)     # URL 제거
no_url = re.sub(r"\s+", " ", no_url).strip()   # 공백 정리
print(no_url)  # "문서: 블로그: 끝."
```
</details>

In [ ]:
# 여기에 작성하세요
text = "문서: https://a.b/c?x=1  블로그: http://blog.com/post  끝."

#### 문제 6. 전화번호 마스킹
문장에서 한국식 전화번호(예: `010-1234-5678`, `02-345-6789`)의 **가운데/끝 4자리**를 `*`로 마스킹하세요.

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 문자열을 다루기 쉬운 형태로 정리합니다.
# 필요한 값만 추출하거나 비교합니다.

import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "문의: 010-1234-5678 / 대리점: 031-987-6543 / 회사: 02-345-6789"  # 실습할 문자열을 준비합니다.
masked = re.sub(r"\b(\d{2,3})-(\d{3,4})-(\d{4})\b", r"\1-****-****", text)  # 정규표현식으로 문자열을 치환합니다.
print(masked)  # 결과를 화면에 출력합니다.
# "문의: 010-****-**** / 대리점: 031-****-**** / 회사: 02-****-****"

``` 
</details>



In [ ]:
# 여기에 작성하세요
text = "문의: 010-1234-5678 / 대리점: 031-987-6543 / 회사: 02-345-6789"

#### 문제 7. 문장 분리 (간단 버전)
`.` `?` `!` 중 하나로 끝나는 구두점 기준으로 **문장을 분리**하고, 출력 시 앞뒤 공백을 제거하세요.

<details> <summary>정답 보기</summary>

```python 
import re  # 정규표현식을 사용하기 위한 모듈입니다.
text = "안녕하세요! 텍스트 분석 재밌죠? 지금은 정규표현식 중입니다.  예시를 더 볼까요!"
sentences = re.split(r"[.!?]+", text)  # 문자열을 기준에 따라 나눕니다.
sentences = [s.strip() for s in sentences if s.strip()]  # 양쪽 공백을 제거합니다.
print(sentences)  # 문자열을 정수로 바꿉니다.
# ['안녕하세요', '텍스트 분석 재밌죠', '지금은 정규표현식 중입니다', '예시를 더 볼까요']
```
</details>

In [ ]:
# 여기에 작성하세요
text = "안녕하세요! 텍스트 분석 재밌죠? 지금은 정규표현식 중입니다.  예시를 더 볼까요!"

### 2.4 토큰화(Tokenization)

토큰화(Tokenization)는 **텍스트를 분석하기 좋은 작은 단위(토큰, token)로 나누는 과정**입니다.  
텍스트마이닝에서는 토큰을 기준으로 단어 빈도, 키워드, 감성 단어, 문서별 특징을 계산합니다.

예를 들어 리뷰 문장 `배송은 빠른데 포장이 아쉬워요`를 분석한다면, 다음과 같은 단위가 필요합니다.

- 문장 전체: `배송은 빠른데 포장이 아쉬워요`
- 띄어쓰기 단위: `배송은`, `빠른데`, `포장이`, `아쉬워요`
- 의미 중심 단위: `배송`, `빠르다`, `포장`, `아쉽다`

여기서는 복잡한 토크나이저보다, 텍스트마이닝에서 자주 쓰는 **띄어쓰기 기준 분리**와 **한국어 형태소 분석**에 집중합니다.


#### 왜 토큰화가 중요한가?

토큰화 결과가 달라지면 분석 결과도 달라집니다.

- `배송이`, `배송은`, `배송도`를 모두 다른 단어로 보면 배송 관련 의견이 흩어질 수 있습니다.
- `환불 요청`, `배송 지연`처럼 함께 등장하는 표현을 보면 단어 하나만 볼 때보다 이슈를 더 잘 이해할 수 있습니다.
- 조사, 접속사, 특수문자처럼 분석에 큰 의미가 없는 토큰은 제거하는 편이 키워드 해석에 도움이 됩니다.

즉, 토큰화는 단순히 문장을 자르는 작업이 아니라 **분석할 단위를 정하는 작업**입니다.


#### 토큰 단위를 정하는 기준

텍스트마이닝에서는 목적에 따라 토큰 단위를 다르게 잡습니다.

| 기준 | 예시 | 적합한 상황 |
|------|------|-------------|
| 띄어쓰기 | `배송은`, `빠른데`, `아쉬워요` | 빠르게 문장을 나눠보고 싶을 때 |
| 명사 추출 | `배송`, `포장`, `환불` | 키워드, 이슈, 주제 파악 |
| 형태소 분석 | `빠르다`, `아쉽다`, `요청하다` | 감성, 행동, 상태 표현까지 보고 싶을 때 |
| n-gram | `배송 지연`, `환불 요청` | 함께 등장하는 표현을 보고 싶을 때 |

너무 잘게 나누면 해석하기 어렵고, 너무 크게 나누면 비슷한 표현이 흩어질 수 있습니다.  
그래서 한국어 텍스트마이닝에서는 명사 추출이나 형태소 분석을 자주 사용합니다.


#### 한국어 형태소 분석: KoNLPy

한국어는 조사와 어미가 단어에 붙어서 의미를 만듭니다.  
띄어쓰기만 기준으로 자르면 `고객이`, `고객은`, `고객에게`가 서로 다른 토큰처럼 처리될 수 있습니다.  

**형태소 분석**은 문장을 더 작은 의미 단위로 나누고, 각 단어의 품사를 함께 확인하는 방법입니다.  
KoNLPy는 한국어 형태소 분석기를 파이썬에서 사용할 수 있게 해주는 라이브러리입니다.

아래 예제는 `Okt` 분석기로 세 가지 결과를 확인합니다.

- `morphs`: 형태소 단위로 나누기
- `nouns`: 명사만 추출하기
- `pos`: 형태소와 품사 태그 함께 보기


In [ ]:
from konlpy.tag import Okt

okt = Okt()

text = "고객이 배송 지연으로 환불을 요청했습니다."

print("형태소:", okt.morphs(text))  # 문자열을 정수로 바꿉니다.
print("명사:", okt.nouns(text))  # 문자열을 정수로 바꿉니다.
print("품사:", okt.pos(text))  # 문자열을 정수로 바꿉니다.


#### 형태소 분석 결과 활용

텍스트 분류나 검색에서는 모든 조사까지 다 쓰기보다, 의미를 많이 담는 명사·동사·형용사를 남기는 방식이 자주 쓰입니다.  
아래 예시는 고객 문의 문장에서 주요 단어만 뽑는 간단한 전처리입니다.


In [ ]:
sentences = [
    "고객이 배송 지연으로 환불을 요청했습니다.",
    "상담사는 주문번호를 확인하고 처리 상태를 안내했습니다.",
    "환불 규정에 따라 영업일 기준 3일 안에 처리됩니다."
]

stopwords = {"이", "가", "을", "를", "은", "는", "으로", "하고", "에", "안에", "의"}

tokenized = []
for sentence in sentences:
    tokens = [
        word for word, tag in okt.pos(sentence, stem=True)
        if tag in ["Noun", "Verb", "Adjective"] and word not in stopwords
    ]
    tokenized.append(tokens)

tokenized


#### 문제 8. KoNLPy로 명사 추출
문장 `"품질 점검 중 센서 오류가 반복적으로 발생했습니다."`에서 명사만 추출하세요.  

<details> <summary>정답 보기</summary>

```python
from konlpy.tag import Okt

okt = Okt()
sentence = "품질 점검 중 센서 오류가 반복적으로 발생했습니다."
nouns = okt.nouns(sentence)
print(nouns)  # 문자열을 정수로 바꿉니다.
# 출력 예시: ['품질', '점검', '중', '센서', '오류']
```
</details>


In [ ]:
# 여기에 정답을 작성하세요
sentence = "품질 점검 중 센서 오류가 반복적으로 발생했습니다."


#### 문제 9. 단어 단위 토큰화
문장 `"오늘은 자연어 처리를 공부한다."`를 **띄어쓰기 기준**으로 토큰화하세요.  

<details> <summary>정답 보기</summary>

```python
# 풀이 흐름: 문자열을 다루기 쉬운 형태로 정리합니다.
# 필요한 값만 추출하거나 비교합니다.

sentence = "오늘은 자연어 처리를 공부한다."  # 분석할 문장을 입력받습니다.
tokens = sentence.split()  # 문자열을 기준에 따라 나눕니다.
print(tokens)  # 결과를 화면에 출력합니다.
# 출력: ['오늘은', '자연어', '처리를', '공부한다.']

```
</details>

In [ ]:
# 여기에 정답을 작성하세요
sentence = "오늘은 자연어 처리를 공부한다."

### 2.5 미니 실습: 리뷰 키워드 확인

토큰화와 형태소 분석을 활용하면 여러 문장에서 자주 등장하는 키워드를 빠르게 확인할 수 있습니다.  
아래 예시는 리뷰 문장에서 명사만 모아 어떤 이슈가 많이 등장하는지 보는 간단한 텍스트마이닝 흐름입니다.


In [ ]:
from collections import Counter

reviews = [
    "배송은 빨랐지만 포장이 조금 아쉬웠어요.",
    "제품 품질이 좋고 가격도 만족스럽습니다.",
    "배송 지연 때문에 고객센터에 문의했습니다.",
    "포장 상태가 좋아서 파손 없이 받았습니다.",
    "가격 대비 품질은 좋은데 배송 안내가 부족했습니다.",
]

nouns = []
for review in reviews:
    nouns.extend([word for word in okt.nouns(review) if len(word) >= 2])

keyword_counts = Counter(nouns)
keyword_counts.most_common(10)


#### 정리

텍스트마이닝에서는 원문을 바로 모델에 넣기보다 분석하기 쉬운 형태로 준비합니다.

- 수집: 분석할 텍스트를 웹, 파일, 설문 등에서 확보합니다.
- 정제: 불필요한 문자, HTML 태그, 중복 공백 같은 노이즈를 줄입니다.
- 토큰화: 문장을 단어, 형태소, n-gram처럼 분석 가능한 단위로 나눕니다.
